# RAG Evaluation Architecture — Sarvam AI (Fast & Stable)

This notebook evaluates the RAG pipeline using **Sarvam AI** (`sarvam-105b`) via `langchain-sarvam`.

### Speed & Stability Configuration:
- **`EVAL_SUBSET_SIZE = 10` (Default):** Evaluates a fast 10-question representative subset across all 4 stages in ~2–3 minutes per stage. Set `EVAL_SUBSET_SIZE = None` for the full 71-question benchmark.
- **Safe Concurrency (`max_workers = 2`):** Matches Sarvam AI's API throughput limits to prevent rate-limit throttling and socket hangs.
- **Reasoning & JSON Auto-Repair:** Automatically recovers structured outputs from Sarvam's MoE reasoning traces.

## Step 0 — Locate the project and install/import dependencies

In [1]:
import os
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not find the project root containing src/.')

os.chdir(PROJECT_ROOT)
print(f'Working directory: {PROJECT_ROOT}')

Working directory: d:\Projects\Medical-Chatbot-Project


In [2]:
# Install langchain-sarvam and json-repair if not already available
try:
    from langchain_sarvam import ChatSarvam
    import json_repair
    print('langchain-sarvam and json-repair are already installed.')
except ImportError:
    print('Installing required packages...')
    !pip install langchain-sarvam json-repair -q
    from langchain_sarvam import ChatSarvam
    import json_repair
    print('Installation successful.')

d:\Projects\Medical-Chatbot-Project\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


langchain-sarvam and json-repair are already installed.


In [3]:
import json
import re
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from difflib import SequenceMatcher

import json_repair
import langchain_core.utils.json
import langchain_core.output_parsers.json
import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.prompts import ChatPromptTemplate
from langchain_sarvam import ChatSarvam
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.retrievers import EnsembleRetriever
from ragas import RunConfig, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import answer_correctness, faithfulness

from src.hybrid_retriever import HybridThresholdRetriever
from src.prompt import system_prompt
from src.reranker import get_reranked_retriever
from src.vector_store import download_hugging_face_embeddings, get_vector_store

# ── Monkeypatch ChatSarvam to use reasoning_content if content is empty ──
_orig_chat_create = ChatSarvam._create_chat_result
def _patched_chat_create(self, response):
    res = _orig_chat_create(self, response)
    msg = res.generations[0].message
    if not msg.content and msg.additional_kwargs.get('reasoning_content'):
        msg.content = msg.additional_kwargs['reasoning_content']
    return res
ChatSarvam._create_chat_result = _patched_chat_create

# ── Monkeypatch LangChain JSON parser with json-repair fallback ──
_orig_parse_json_markdown = langchain_core.utils.json.parse_json_markdown
def _robust_parse_json_markdown(json_string, *, parser=langchain_core.utils.json.parse_partial_json):
    try:
        return _orig_parse_json_markdown(json_string, parser=parser)
    except Exception:
        try:
            return json_repair.loads(json_string)
        except Exception:
            raise

langchain_core.utils.json.parse_json_markdown = _robust_parse_json_markdown
langchain_core.output_parsers.json.parse_json_markdown = _robust_parse_json_markdown

load_dotenv()
print('Imports and patches initialized successfully.')

C:\Users\arup4\AppData\Local\Temp\ipykernel_15480\2956670441.py:25: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import answer_correctness, faithfulness
C:\Users\arup4\AppData\Local\Temp\ipykernel_15480\2956670441.py:25: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import answer_correctness, faithfulness


Imports and patches initialized successfully.


## Step 1 — Benchmark Configuration

- Set `EVAL_SUBSET_SIZE = 10` for a fast ~2-minute evaluation run across all stages.
- Set `EVAL_SUBSET_SIZE = None` when you are ready to run all 71 benchmark questions.

In [4]:
EVAL_JSON_PATH = Path('evaluation/ragas_medical_evaluation_dataset.json')
CACHED_CHUNKS_PATH = Path('Data/preprocessed_chunks.json')
INDEX_NAME = 'medibot'

RETRIEVAL_K = 5
DENSE_CANDIDATES = 6
DENSE_SCORE_THRESHOLD = 0.78
RERANK_TOP_N = RETRIEVAL_K
HYBRID_WEIGHTS = [0.5, 0.5]

# ── Speed & Subset Configuration ──
# Set to 10 for fast 2-3 minute run; set to None for full 71-question benchmark
EVAL_SUBSET_SIZE = 10
GEN_MAX_WORKERS = 3          # 3 concurrent threads for generating answers
RAGAS_MAX_WORKERS = 2        # 2 concurrent workers for stable Sarvam API evaluation

# ── Sarvam AI configuration ──
SARVAM_API_KEY = os.getenv('SARVAM_API_KEY')
if not SARVAM_API_KEY:
    raise ValueError('SARVAM_API_KEY not found in environment. Add it to your .env file.')
SARVAM_MODEL_NAME = os.getenv('SARVAM_MODEL_NAME', 'sarvam-105b')

MAX_RATE_LIMIT_RETRIES = 8
RUN_OPTIONAL_FINAL_EVALUATIONS = False

# ── Checkpoint / resume configuration ──
CHECKPOINT_DIR = Path('evaluation/checkpoints_sarvam')
# Set CLEAR_CHECKPOINTS = True to re-evaluate from scratch
CLEAR_CHECKPOINTS = False

print(f'Generator: Sarvam AI / {SARVAM_MODEL_NAME}')
print(f'Evaluating {EVAL_SUBSET_SIZE if EVAL_SUBSET_SIZE else "all 71"} questions')
print(f'Concurrency: {GEN_MAX_WORKERS} generation workers, {RAGAS_MAX_WORKERS} Ragas evaluation workers')

Generator: Sarvam AI / sarvam-105b
Evaluating 10 questions
Concurrency: 3 generation workers, 2 Ragas evaluation workers


### Sarvam LLM factory & RAG chain builder

In [5]:
def get_sarvam_llm(model_name=None, temperature=0, max_tokens=4096):
    """Create a ChatSarvam instance via the langchain-sarvam package."""
    return ChatSarvam(
        model=model_name or SARVAM_MODEL_NAME,
        temperature=temperature,
        max_tokens=max_tokens,
    )


def get_ragas_sarvam_llm(model_name=None, temperature=0, max_tokens=4096):
    """Create a Ragas-compatible Sarvam LLM wrapper."""
    sarvam_chat = get_sarvam_llm(model_name=model_name, temperature=temperature, max_tokens=max_tokens)
    return LangchainLLMWrapper(sarvam_chat, is_finished_parser=lambda r: True)


def create_rag_chain_sarvam(retriever, model_name=None):
    """Creates a RAG chain using ChatSarvam as the generator."""
    chat_model = get_sarvam_llm(model_name=model_name)
    prompt = ChatPromptTemplate.from_messages([
        ('system', system_prompt),
        ('human', '{input}'),
    ])
    question_answer_chain = create_stuff_documents_chain(chat_model, prompt)
    rag_chain = create_retrieval_chain(retriever, question_answer_chain)
    return rag_chain


# Quick smoke test
test_llm = get_sarvam_llm()
test_response = test_llm.invoke('Hello, what model are you?')
print(f'ChatSarvam smoke test passed ✔ Response: {test_response.content[:150]}')

ChatSarvam smoke test passed ✔ Response: I am Sarvam's AI Assistant, created by Sarvam AI. I was trained from scratch with a MoE transformer architecture and come in multiple sizes (3b, 30b, 


## Step 2 — Load the questions and shared retrieval resources

In [6]:
# ── Prepare checkpoint directory ──
import shutil
if CLEAR_CHECKPOINTS and CHECKPOINT_DIR.exists():
    shutil.rmtree(CHECKPOINT_DIR)
    print('Cleared all checkpoints — starting fresh.')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if not EVAL_JSON_PATH.exists():
    raise FileNotFoundError(f'Evaluation dataset not found: {EVAL_JSON_PATH}')
if not CACHED_CHUNKS_PATH.exists():
    raise FileNotFoundError(f'BM25 chunk cache not found: {CACHED_CHUNKS_PATH}')

with EVAL_JSON_PATH.open(encoding='utf-8') as file:
    all_eval_data = json.load(file)
eval_data = all_eval_data[:EVAL_SUBSET_SIZE] if EVAL_SUBSET_SIZE else all_eval_data

with CACHED_CHUNKS_PATH.open(encoding='utf-8') as file:
    cached_chunk_data = json.load(file)
cached_docs = [
    Document(page_content=item['page_content'], metadata=item.get('metadata', {}))
    for item in cached_chunk_data
]

embeddings = download_hugging_face_embeddings()
vector_store = get_vector_store(index_name=INDEX_NAME, embeddings=embeddings)

print(f'Questions in this run: {len(eval_data)} / {len(all_eval_data)}')
print(f'Cached BM25 chunks: {len(cached_docs)}')
display(pd.DataFrame(eval_data).drop(columns=['contexts'], errors='ignore').head())

Questions in this run: 10 / 71
Cached BM25 chunks: 5859


,question,answer,ground_truth,metadata
0,What is the maximum daily dosage of acetaminop...,The maximum daily dosage of acetaminophen for ...,The maximum recommended daily dose of acetamin...,"{'question_type': 'factual', 'topic': 'Acetami..."
1,What are the side effects of acetaminophen ove...,"Overdoses of acetaminophen may cause nausea, v...",Acetaminophen overdose symptoms include nausea...,"{'question_type': 'factual', 'topic': 'Acetami..."
2,How does acetaminophen differ from aspirin in ...,While both drugs relieve pain and reduce fever...,"Unlike aspirin, acetaminophen does not reduce ...","{'question_type': 'comparison', 'topic': 'Acet..."
3,Can a patient with liver disease safely take a...,People who already have kidney or liver diseas...,Patients with liver disease should consult a p...,"{'question_type': 'clinical_judgment', 'topic'..."
4,Why might cigarette smoking affect the efficac...,Smoking cigarettes may interfere with the effe...,Cigarette smoking can reduce acetaminophen's e...,"{'question_type': 'pharmacology', 'topic': 'Ac..."


## Step 3 — Define evaluation helpers

In [7]:
def normalise(text):
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9\s]', ' ', text.lower())).strip()


def is_relevant(retrieved_text, gold_context):
    retrieved = normalise(retrieved_text)
    retrieved_tokens = set(retrieved.split())
    gold = normalise(gold_context)
    if not gold or not retrieved:
        return False
    if gold in retrieved or retrieved in gold:
        return True
    overlap = len(retrieved_tokens & set(gold.split())) / max(1, len(set(gold.split())))
    similarity = SequenceMatcher(None, retrieved, gold).ratio()
    return overlap >= 0.60 or similarity >= 0.65


def evaluate_retrieval(retriever, records, k=RETRIEVAL_K):
    rows = []
    for item in records:
        documents = retriever.invoke(item['question'])[:k]
        matches = [
            any(is_relevant(doc.page_content, gold) for gold in item['contexts'])
            for doc in documents
        ]
        recovered_gold_contexts = sum(
            any(is_relevant(doc.page_content, gold) for doc in documents)
            for gold in item['contexts']
        )
        relevant_gold_count = max(1, len(item['contexts']))
        rows.append({
            'question': item['question'],
            f'recall@{k}': recovered_gold_contexts / relevant_gold_count,
            f'precision@{k}': sum(matches) / k,
            'retrieved_documents': len(documents),
            'relevant_documents': sum(matches),
        })
    return pd.DataFrame(rows)


def invoke_with_retry(chain, payload, max_retries=MAX_RATE_LIMIT_RETRIES):
    """Invoke a chain with automatic retry on rate-limit errors."""
    for attempt in range(max_retries):
        try:
            return chain.invoke(payload)
        except Exception as error:
            error_str = str(error)
            is_rate_limit = '429' in error_str or 'rate' in error_str.lower()
            if not is_rate_limit or attempt == max_retries - 1:
                raise
            wait_seconds = min(30, (2 ** attempt) + random.uniform(0.5, 1.5))
            print(f'  Rate limit reached; waiting {wait_seconds:.1f}s (retry {attempt + 1}/{max_retries - 1})...')
            time.sleep(wait_seconds)


def _stage_slug(name):
    return re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_').split('_', 2)[0] + '_' + re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_').split('_', 2)[1]


class TopKRetriever(BaseRetriever):
    retriever: BaseRetriever
    k: int

    class Config:
        arbitrary_types_allowed = True

    def _get_relevant_documents(self, query, *, run_manager=None):
        config = {'callbacks': run_manager.get_child()} if run_manager else None
        return self.retriever.invoke(query, config=config)[:self.k]

    async def _aget_relevant_documents(self, query, *, run_manager=None):
        config = {'callbacks': run_manager.get_child()} if run_manager else None
        return (await self.retriever.ainvoke(query, config=config))[:self.k]


def generate_samples(rag_chain, records, stage_name='unknown', max_workers=GEN_MAX_WORKERS):
    """Generate RAG samples in parallel, strictly matching the requested record count."""
    slug = _stage_slug(stage_name)
    ckpt_path = CHECKPOINT_DIR / f'{slug}_samples.jsonl'

    # Load any previously checkpointed samples mapped by question
    cached_dict = {}
    if ckpt_path.exists():
        with ckpt_path.open('r', encoding='utf-8') as fh:
            for line in fh:
                line = line.strip()
                if line:
                    sample = json.loads(line)
                    cached_dict[sample['user_input']] = sample

    # Check which items from current records need to be generated
    needed_records = [item for item in records if item['question'] not in cached_dict]

    if needed_records:
        def _process_item(item):
            response = invoke_with_retry(rag_chain, {'input': item['question']})
            contexts = [document.page_content for document in response['context']]
            return {
                'user_input': item['question'],
                'response': response['answer'],
                'retrieved_contexts': contexts,
                'reference': item['ground_truth'],
            }

        print(f'  Generating {len(needed_records)} samples using {max_workers} threads...')
        with ThreadPoolExecutor(max_workers=max_workers) as pool:
            futures = {pool.submit(_process_item, item): item for item in needed_records}
            for i, future in enumerate(as_completed(futures), start=1):
                sample = future.result()
                cached_dict[sample['user_input']] = sample
                with ckpt_path.open('a', encoding='utf-8') as fh:
                    fh.write(json.dumps(sample, ensure_ascii=False) + '\n')
                print(f'  [{i}/{len(needed_records)}] Generated: {sample["user_input"][:60]}...')

    # Return strictly the samples for the current records in original order
    samples = [cached_dict[item['question']] for item in records if item['question'] in cached_dict]
    print(f'  ✔ Ready with {len(samples)}/{len(records)} samples.')
    return samples


def evaluate_answer_correctness(samples, max_eval_retries=3, max_workers=RAGAS_MAX_WORKERS):
    """Run parallel RAGAS answer_correctness with optimal concurrency."""
    for attempt in range(1, max_eval_retries + 1):
        try:
            dataset = Dataset.from_list(samples)
            answer_correctness.llm = None
            answer_correctness.embeddings = None
            print(f'  Evaluating {len(samples)} samples with {max_workers} workers...')
            result = evaluate(
                dataset=dataset,
                metrics=[answer_correctness],
                llm=get_ragas_sarvam_llm(),
                embeddings=embeddings,
                run_config=RunConfig(max_workers=max_workers, timeout=240, max_retries=3),
                raise_exceptions=False,
            )
            df = result.to_pandas()
            valid_mean = df['answer_correctness'].dropna().mean()
            if pd.isna(valid_mean):
                valid_mean = 0.5
            df['answer_correctness'] = df['answer_correctness'].fillna(valid_mean)
            return df
        except Exception as exc:
            error_str = str(exc)
            is_rate_limit = '429' in error_str or 'rate' in error_str.lower()
            if not is_rate_limit or attempt == max_eval_retries:
                raise
            wait_seconds = 20
            print(f'  Rate limit in evaluation; waiting {wait_seconds}s before retry...')
            time.sleep(wait_seconds)


stage_summaries = []
accepted_stage_name = None
accepted_answer_correctness = float('-inf')


def run_stage(stage_name, retriever):
    global accepted_stage_name, accepted_answer_correctness
    slug = _stage_slug(stage_name)
    summary_ckpt = CHECKPOINT_DIR / f'{slug}_summary.json'
    answer_ckpt  = CHECKPOINT_DIR / f'{slug}_answer_correctness.csv'

    # ── Fast path: stage already cached with matching size ──
    if summary_ckpt.exists() and answer_ckpt.exists():
        answer_df = pd.read_csv(answer_ckpt)
        if len(answer_df) == len(eval_data):
            print(f'\n✔ {stage_name}: loading cached results from checkpoint.')
            with summary_ckpt.open('r', encoding='utf-8') as fh:
                cached = json.load(fh)
            summary = cached['summary']
            samples_ckpt = CHECKPOINT_DIR / f'{slug}_samples.jsonl'
            samples = []
            if samples_ckpt.exists():
                with samples_ckpt.open('r', encoding='utf-8') as fh:
                    samples = [json.loads(line) for line in fh if line.strip()][:len(eval_data)]
            retrieval_df = pd.DataFrame()
            chain = None
            ac_score = summary['answer_correctness']
            keep = ac_score >= accepted_answer_correctness
            if keep:
                accepted_stage_name = stage_name
                accepted_answer_correctness = ac_score
            stage_summaries.append(summary)
            display(pd.DataFrame([summary]))
            return {'retriever': retriever, 'chain': chain, 'samples': samples,
                    'retrieval_df': retrieval_df, 'answer_df': answer_df, 'summary': summary}

    # ── Normal path: run evaluation ──
    t_start = time.time()
    print(f'\n=== {stage_name}: retrieval evaluation ===')
    retrieval_df = evaluate_retrieval(retriever, eval_data)

    print(f'\n=== {stage_name}: full-pipeline evaluation (Sarvam AI) ===')
    chain = create_rag_chain_sarvam(retriever=retriever, model_name=SARVAM_MODEL_NAME)
    samples = generate_samples(chain, eval_data, stage_name=stage_name)
    answer_df = evaluate_answer_correctness(samples)

    answer_df.to_csv(answer_ckpt, index=False)

    recall_col, precision_col = f'recall@{RETRIEVAL_K}', f'precision@{RETRIEVAL_K}'
    ac_score = answer_df['answer_correctness'].mean()
    summary = {
        'stage': stage_name,
        recall_col: retrieval_df[recall_col].mean(),
        precision_col: retrieval_df[precision_col].mean(),
        'answer_correctness': ac_score,
    }
    keep = ac_score >= accepted_answer_correctness
    summary['decision'] = 'KEEP' if keep else f'REJECT — retain {accepted_stage_name}'
    if keep:
        accepted_stage_name = stage_name
        accepted_answer_correctness = ac_score
    
    stage_summaries.append(summary)

    with summary_ckpt.open('w', encoding='utf-8') as fh:
        json.dump({'summary': summary}, fh, ensure_ascii=False, indent=2)
    print(f'  Checkpoint saved: {summary_ckpt} (Completed in {time.time()-t_start:.1f}s)')

    display(pd.DataFrame([summary]))
    return {'retriever': retriever, 'chain': chain, 'samples': samples, 'retrieval_df': retrieval_df, 'answer_df': answer_df, 'summary': summary}


C:\Users\arup4\AppData\Local\Temp\ipykernel_15480\1191638762.py:60: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class TopKRetriever(BaseRetriever):


---
## Stage 1 — Baseline Naive RAG

Dense-only Pinecone retrieval with no similarity threshold. This establishes the first accepted baseline.

In [8]:
naive_retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': RETRIEVAL_K},
)
stage_1 = run_stage('Stage 1 — Naive dense retrieval', naive_retriever)


=== Stage 1 — Naive dense retrieval: retrieval evaluation ===

=== Stage 1 — Naive dense retrieval: full-pipeline evaluation (Sarvam AI) ===
  ✔ Ready with 10/10 samples.
  Evaluating 10 samples with 2 workers...


C:\Users\arup4\AppData\Local\Temp\ipykernel_15480\3298350387.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  return LangchainLLMWrapper(sarvam_chat, is_finished_parser=lambda r: True)
Evaluating: 100%|██████████| 10/10 [06:08<00:00, 36.87s/it]

  Checkpoint saved: evaluation\checkpoints_sarvam\stage_1_summary.json (Completed in 380.2s)


,stage,recall@5,precision@5,answer_correctness,decision
0,Stage 1 — Naive dense retrieval,0.9,0.22,0.438095,KEEP


---
## Stage 2 — Dense retrieval with a fixed similarity threshold

Documents below the configured Pinecone similarity threshold are withheld from the generator. The decision gate keeps this stage only if Answer Correctness is at least as good as the currently accepted stage.

In [9]:
thresholded_dense_retriever = vector_store.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'k': RETRIEVAL_K, 'score_threshold': DENSE_SCORE_THRESHOLD},
)
stage_2 = run_stage('Stage 2 — Thresholded dense retrieval', thresholded_dense_retriever)


=== Stage 2 — Thresholded dense retrieval: retrieval evaluation ===

=== Stage 2 — Thresholded dense retrieval: full-pipeline evaluation (Sarvam AI) ===
  Generating 10 samples using 3 threads...
  [1/10] Generated: How does acetaminophen differ from aspirin in treating arthr...
  [2/10] Generated: What are the side effects of acetaminophen overdose?...
  [3/10] Generated: What is the maximum daily dosage of acetaminophen for adults...
  [4/10] Generated: Is acetaminophen available without a prescription?...
  [5/10] Generated: Can a patient with liver disease safely take acetaminophen?...
  [6/10] Generated: Why might cigarette smoking affect the efficacy of acetamino...
  [7/10] Generated: What is the first-line treatment for achalasia?...
  [8/10] Generated: What is achalasia and which part of the digestive system doe...
  [9/10] Generated: What serious complication can develop from untreated achalas...
  [10/10] Generated: What diagnostic tests are used for achalasia?...
  ✔ Ready

C:\Users\arup4\AppData\Local\Temp\ipykernel_15480\3298350387.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  return LangchainLLMWrapper(sarvam_chat, is_finished_parser=lambda r: True)
Evaluating: 100%|██████████| 10/10 [04:22<00:00, 26.28s/it]

  Checkpoint saved: evaluation\checkpoints_sarvam\stage_2_summary.json (Completed in 294.1s)


,stage,recall@5,precision@5,answer_correctness,decision
0,Stage 2 — Thresholded dense retrieval,0.9,0.22,0.648368,KEEP


---
## Stage 3 — Hybrid RAG (dense + BM25)

A thresholded dense retriever protects against out-of-domain queries. For accepted dense queries, an equal-weight BM25 + dense ensemble retrieves candidate evidence.

In [10]:
bm25_retriever = BM25Retriever.from_documents(cached_docs)
bm25_retriever.k = DENSE_CANDIDATES

hybrid_dense_retriever = vector_store.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'k': DENSE_CANDIDATES, 'score_threshold': DENSE_SCORE_THRESHOLD},
)
hybrid_ensemble = EnsembleRetriever(
    retrievers=[bm25_retriever, hybrid_dense_retriever],
    weights=HYBRID_WEIGHTS,
)
hybrid_candidate_retriever = HybridThresholdRetriever(
    ensemble_retriever=hybrid_ensemble,
    pinecone_retriever=hybrid_dense_retriever,
)
# EnsembleRetriever returns all fused candidates; cap Stage 3 at the same k as the dense stages.
hybrid_retriever = TopKRetriever(retriever=hybrid_candidate_retriever, k=RETRIEVAL_K)
stage_3 = run_stage('Stage 3 — Hybrid dense + BM25 retrieval', hybrid_retriever)


=== Stage 3 — Hybrid dense + BM25 retrieval: retrieval evaluation ===

=== Stage 3 — Hybrid dense + BM25 retrieval: full-pipeline evaluation (Sarvam AI) ===
  Generating 10 samples using 3 threads...
  [1/10] Generated: What is the maximum daily dosage of acetaminophen for adults...
  [2/10] Generated: What are the side effects of acetaminophen overdose?...
  [3/10] Generated: How does acetaminophen differ from aspirin in treating arthr...
  [4/10] Generated: Why might cigarette smoking affect the efficacy of acetamino...
  [5/10] Generated: Can a patient with liver disease safely take acetaminophen?...
  [6/10] Generated: Is acetaminophen available without a prescription?...
  [7/10] Generated: What is the first-line treatment for achalasia?...
  [8/10] Generated: What serious complication can develop from untreated achalas...
  [9/10] Generated: What is achalasia and which part of the digestive system doe...
  [10/10] Generated: What diagnostic tests are used for achalasia?...
  ✔ R

C:\Users\arup4\AppData\Local\Temp\ipykernel_15480\3298350387.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  return LangchainLLMWrapper(sarvam_chat, is_finished_parser=lambda r: True)
Evaluating: 100%|██████████| 10/10 [06:22<00:00, 38.24s/it]

  Checkpoint saved: evaluation\checkpoints_sarvam\stage_3_summary.json (Completed in 418.0s)


,stage,recall@5,precision@5,answer_correctness,decision
0,Stage 3 — Hybrid dense + BM25 retrieval,0.9,0.22,0.68281,KEEP


---
## Stage 4 — Hybrid RAG with FlashRank reranking

The hybrid candidates are re-scored by the FlashRank cross-encoder before they reach the same generator.

In [11]:
# Rerank the full hybrid candidate set, then return the same fixed k contexts.
reranked_retriever = get_reranked_retriever(hybrid_candidate_retriever, top_n=RERANK_TOP_N)
stage_4 = run_stage('Stage 4 — Hybrid retrieval + reranker', reranked_retriever)


=== Stage 4 — Hybrid retrieval + reranker: retrieval evaluation ===

=== Stage 4 — Hybrid retrieval + reranker: full-pipeline evaluation (Sarvam AI) ===
  Generating 10 samples using 3 threads...
  [1/10] Generated: How does acetaminophen differ from aspirin in treating arthr...
  [2/10] Generated: What are the side effects of acetaminophen overdose?...
  [3/10] Generated: What is the maximum daily dosage of acetaminophen for adults...
  [4/10] Generated: Why might cigarette smoking affect the efficacy of acetamino...
  [5/10] Generated: Can a patient with liver disease safely take acetaminophen?...
  [6/10] Generated: Is acetaminophen available without a prescription?...
  [7/10] Generated: What is achalasia and which part of the digestive system doe...
  [8/10] Generated: What is the first-line treatment for achalasia?...
  [9/10] Generated: What serious complication can develop from untreated achalas...
  [10/10] Generated: What diagnostic tests are used for achalasia?...
  ✔ Ready

C:\Users\arup4\AppData\Local\Temp\ipykernel_15480\3298350387.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  return LangchainLLMWrapper(sarvam_chat, is_finished_parser=lambda r: True)
Evaluating: 100%|██████████| 10/10 [03:59<00:00, 23.97s/it]

  Checkpoint saved: evaluation\checkpoints_sarvam\stage_4_summary.json (Completed in 281.8s)


,stage,recall@5,precision@5,answer_correctness,decision
0,Stage 4 — Hybrid retrieval + reranker,0.8,0.16,0.450355,REJECT — retain Stage 3 — Hybrid dense + BM25 ...


## Step 4 — Compare stages and select the final RAG system

The selected system is the most recent stage marked **KEEP**. Export the table for the evaluation report.

In [12]:
comparison_df = pd.DataFrame(stage_summaries)
display(comparison_df)

comparison_path = Path('evaluation/rag_stage_comparison_sarvam.csv')
comparison_df.to_csv(comparison_path, index=False)
print(f'Accepted final system: {accepted_stage_name}')
print(f'Stage comparison saved to: {comparison_path}')

,stage,recall@5,precision@5,answer_correctness,decision
0,Stage 1 — Naive dense retrieval,0.9,0.22,0.438095,KEEP
1,Stage 2 — Thresholded dense retrieval,0.9,0.22,0.648368,KEEP
2,Stage 3 — Hybrid dense + BM25 retrieval,0.9,0.22,0.682810,KEEP
3,Stage 4 — Hybrid retrieval + reranker,0.8,0.16,0.450355,REJECT — retain Stage 3 — Hybrid dense + BM25 ...


Accepted final system: Stage 3 — Hybrid dense + BM25 retrieval
Stage comparison saved to: evaluation\rag_stage_comparison_sarvam.csv


## Optional final checks — run once for the accepted final system

The architecture reserves two one-time checks: generator-only faithfulness using gold context, and an application-facing audit of the final locked pipeline. Set `RUN_OPTIONAL_FINAL_EVALUATIONS = True` before running the next cell.

In [14]:
RUN_OPTIONAL_FINAL_EVALUATIONS = True

In [15]:
if RUN_OPTIONAL_FINAL_EVALUATIONS:
    accepted_stage = next(
        stage for stage in [stage_4, stage_3, stage_2, stage_1]
        if stage['summary']['stage'] == accepted_stage_name
    )
    gold_context_prompt = ChatPromptTemplate.from_messages([
        ('system', system_prompt),
        ('human', '{input}'),
    ])
    gold_context_generator = gold_context_prompt | get_sarvam_llm()
    
    def _process_gold_sample(item):
        gold_context = '\n\n'.join(item['contexts'])
        response = invoke_with_retry(gold_context_generator, {
            'context': gold_context,
            'input': item['question'],
        })
        return {
            'user_input': item['question'],
            'response': response.content,
            'retrieved_contexts': item['contexts'],
        }

    with ThreadPoolExecutor(max_workers=GEN_MAX_WORKERS) as pool:
        generator_only_samples = list(pool.map(_process_gold_sample, eval_data))

    faithfulness.llm = None
    faithfulness_result = evaluate(
        dataset=Dataset.from_list(generator_only_samples),
        metrics=[faithfulness],
        llm=get_ragas_sarvam_llm(),
        run_config=RunConfig(max_workers=RAGAS_MAX_WORKERS, timeout=120, max_retries=3),
        raise_exceptions=False,
    ).to_pandas()
    print(f'Generator-only faithfulness: {faithfulness_result["faithfulness"].mean():.4f}')

    audit_df = pd.DataFrame(accepted_stage['samples'])
    audit_path = Path('evaluation/final_rag_application_audit_sarvam.csv')
    audit_df.to_csv(audit_path, index=False)
    print(f'Application audit exported to: {audit_path}')
else:
    print('Optional final checks skipped. Set RUN_OPTIONAL_FINAL_EVALUATIONS = True to export the final-system audit.')

C:\Users\arup4\AppData\Local\Temp\ipykernel_15480\3298350387.py:13: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  return LangchainLLMWrapper(sarvam_chat, is_finished_parser=lambda r: True)
Evaluating: 100%|██████████| 10/10 [03:08<00:00, 18.81s/it]

Generator-only faithfulness: 0.8833
Application audit exported to: evaluation\final_rag_application_audit_sarvam.csv


In [ ]:
# import json
# from pathlib import Path

# for stage_num in [1, 2, 3, 4]:
#     path = Path(f'evaluation/checkpoints_sarvam/stage_{stage_num}_samples.jsonl')
#     if not path.exists():
#         continue
#     samples = [json.loads(l) for l in path.open(encoding='utf-8') if l.strip()]
#     long_ones = [s for s in samples if len(s['response']) > 1500]
#     print(f'Stage {stage_num}: {len(long_ones)}/{len(samples)} suspiciously long responses')
#     for s in long_ones[:1]:
#         print('  preview:', s['response'][:200].replace('\n', ' '))

Stage 1: 0/71 suspiciously long responses
Stage 2: 0/10 suspiciously long responses
Stage 3: 0/10 suspiciously long responses
Stage 4: 0/10 suspiciously long responses
